# Sample Lists
- checking raw seqs 
- creating different sample lists that fit different conditions to use in assembly set
- purpose of this is to try assembling with similar sample types to get better results 

In [2]:
import numpy as np
import pandas as pd
import os
import re

In [3]:
os.chdir('/scratch/workspace/brooke_sienkiewicz_student_uml_edu-brooke-belseq/metadata')

In [4]:
ls

GW_Genohub_libraryprepped_submission_5799597.xlsx
GW_Genohub_submission_5799597.xlsx
Metagenomics_Tracker_Belize.csv


In [5]:
sample_data=pd.read_csv('Metagenomics_Tracker_Belize.csv', index_col=0)

In [6]:
sample_data.head()

,Health_Status,Starting_Weight,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_ng_ul,Microbe_Location,Microbe_clean_date/n,Host_ng_ul,Host_Location,...,Notes,Seq_date,Host_Seq_date,Microbe_seq_file,Host_seq_file,Seq_Location (in Unity),Sample_physical_location,Extraction_physical_location,Location_notes,Sample_Code = datecollected_tagnumber_transect_samplenumber_species
052022_BEL_CBC_T2_45_PSTR,Diseased_Margin,181,10_10_2023,22.8,10_17_2023,7.5,UML_NARWHAL_R2_B1,y,0.772,UML_NARWHAL_R6_B2,...,NaN,01_24; 10_19_23,NaN,052022_BEL_CBC_T2_45_PSTR,NaN,/project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_...,NaN,UML_NARWHAL_R2_B3,NaN,NaN
052022_BEL_CBC_T2_46_PSTR,Diseased_Tissue,162,10_10_2023,63.9,10_17_2023,18.5,UML_NARWHAL_R2_B1,y,too low,UML_NARWHAL_R6_B2,...,NaN,01_24; 10_19_23,NaN,052022_BEL_CBC_T2_46_PSTR,NaN,/project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_...,NaN,UML_NARWHAL_R2_B3,NaN,NaN
052022_BEL_CBC_T2_59_OFAV,Healthy,174,10_3_2023,73.7,10_16_2023,11.7,UML_NARWHAL_R2_B1,y,NaN,UML_NARWHAL_R6_B2,...,NaN,01_24; 10_19_23,NaN,052022_BEL_CBC_T2_59_OFAV,NaN,/project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_...,NaN,UML_NARWHAL_R2_B3,NaN,NaN
042024_BEL_CBC_T1_925_PAST,Healthy,325.5,8_19_2024,5.46,9_28_2024,1.57,UML_NARWHAL_R6_B30,9_28_2024,0.212,UML_NARWHAL_R6_B31,...,NaN,NaN,NaN,NaN,NaN,NaN,UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B29,AS - next to PAST 19,NaN
052022_BEL_CBC_T2_72_OFAV,Healthy,75,10_3_2023,47.6,10_11_23,3.96,UML_NARWHAL_R6_B1,10_12_23,0.522,UML_NARWHAL_R6_B2,...,NaN,01_24; 10_19_23,NaN,052022_BEL_CBC_T2_72_OFAV,NaN,/project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_...,NaN,UML_NARWHAL_R2_B3,NaN,NaN


In [7]:
# make a column for year 

# Extract year from the sample names
def extract_year(sample_name):
    return sample_name.split('_')[0][2:]

# Create a new column 'Year' using the 'apply' function
sample_data['Year'] = sample_data.index.to_series().apply(extract_year)

# Print the updated DataFrame
print(sample_data['Year'])

052022_BEL_CBC_T2_45_PSTR      2022
052022_BEL_CBC_T2_46_PSTR      2022
052022_BEL_CBC_T2_59_OFAV      2022
042024_BEL_CBC_T1_925_PAST     2024
052022_BEL_CBC_T2_72_OFAV      2022
                               ... 
082024_BEL_CBC_T3_1561_MCAV    2024
082024_BEL_CBC_T3_1563_SSID    2024
082024_BEL_CBC_T3_1564_PAST    2024
052022_BEL_CBC_T3_16_SSID      2022
092023_BEL_CBC_T1_175_PAST     2023
Name: Year, Length: 524, dtype: object


In [8]:
# make column for species 
def extract_species(sample_name):
    return sample_name.split('_')[-1]  # Extract last element after splitting by "_"

sample_data['Species'] = sample_data.index.to_series().apply(extract_species)
print(sample_data['Species'])

052022_BEL_CBC_T2_45_PSTR      PSTR
052022_BEL_CBC_T2_46_PSTR      PSTR
052022_BEL_CBC_T2_59_OFAV      OFAV
042024_BEL_CBC_T1_925_PAST     PAST
052022_BEL_CBC_T2_72_OFAV      OFAV
                               ... 
082024_BEL_CBC_T3_1561_MCAV    MCAV
082024_BEL_CBC_T3_1563_SSID    SSID
082024_BEL_CBC_T3_1564_PAST    PAST
052022_BEL_CBC_T3_16_SSID      SSID
092023_BEL_CBC_T1_175_PAST     PAST
Name: Species, Length: 524, dtype: object


In [9]:
# remove unnecessary columns
columns_to_drop = ["Sample_Code = datecollected_tagnumber_transect_samplenumber_species",
                   'Date_Libprep','Host_Seq_date','Host_seq_file','Sample_physical_location','Extraction_physical_location','Location_notes']
sample_data = sample_data.drop(columns=columns_to_drop)
#print(sample_data.columns)

## samples sequenced in this run 
- make list of samples 
- make sure we have all seqs to match these samples 
- sequence date: nov 2024
- received seqs back: 03/17/2025 

In [10]:
# replace na with 'no'
sample_data['Seq_date']=sample_data['Seq_date'].fillna('no')
samples24=sample_data.loc[sample_data['Seq_date'].str.contains('24')]

In [11]:
# arrange by date and species
samples24=samples24.sort_values(by=['Year','Species'])

In [12]:
# export list of samples sequenced
samplelist24 = samples24.index.tolist()

In [13]:
# view lines 
samplelist24[20:25]

['062019_BEL_CBC_T3_16_MCAV',
 '062019_BEL_CBC_T3_6_MCAV',
 '062019_BEL_CBC_T3_8_MCAV',
 '062019_BEL_CBC_T3_9_MCAV',
 '062019_BEL_CBC_T1_10_MMEA']

### crosscheck seq files 

In [14]:
### crosscheck genohub submission form 
# samplelist24 - from metagenomics tracker
# genohublist24 - from genohub submission forms 

In [15]:
cd ..

/scratch/workspace/brooke_sienkiewicz_student_uml_edu-brooke-belseq


In [16]:
ls

03172025/          md5checksum             slurm-aws-32614617.out
aws                metadata/               slurm-checksum-30446035.out
genohublist24.txt  sample_list             slurm-checksum-32897719.out
md5_checksums.txt  slurm-aws-30186235.out


In [17]:
with open("genohublist24.txt", "r") as f:
    genohublist24 = [line.strip() for line in f]  # Removes extra spaces or newlines

print(genohublist24)
print(len(genohublist24)) 

['062019_BEL_CBC_T3_25_PAST', '122022_BEL_CBC_T1_133_PSTR', '122022_BEL_CBC_T2_116_PSTR', '122022_BEL_CBC_T4_35_PSTR', '122022_BEL_CBC_T2_99_PSTR', '122022_BEL_CBC_T1_123_OANN', '052022_BEL_CBC_T1_63_OFAV', '062019_BEL_CBC_T3_4_PAST', '062019_BEL_CBC_T1_10_MMEA', '062019_BEL_CBC_T1_14_MMEA', '062019_BEL_CBC_T2_12_MMEA', '062019_BEL_CBC_T2_13_MMEA', '052022_BEL_CBC_T1_39_MCAV', '052022_BEL_CBC_T1_54_MCAV', '052022_BEL_CBC_T1_62_MCAV', '052022_BEL_CBC_T2_12_MCAV', '052022_BEL_CBC_T2_4_MCAV', '052022_BEL_CBC_T2_56_MCAV', '122022_BEL_CBC_T1_151_MCAV', '122022_BEL_CBC_T2_86_MCAV', '122022_BEL_CBC_T2_88_MCAV', '122022_BEL_CBC_T2_92_MCAV', '122022_BEL_CBC_T2_95_MCAV', '122022_BEL_CBC_T3_119_MCAV', '122022_BEL_CBC_T3_128_MCAV', '122022_BEL_CBC_T3_142_MCAV', '122022_BEL_CBC_T3_155_MCAV', '122022_BEL_CBC_T4_3_MCAV', '062019_BEL_CBC_T2_14_MMEA', '062019_BEL_CBC_T2_15_MMEA', '122022_BEL_CBC_T4_1_OFAV', '122022_BEL_CBC_T3_133_MCAV', '122022_BEL_CBC_T4_14_MCAV', '122022_BEL_CBC_T4_5_MCAV', '052022_B

In [18]:
print(samplelist24)
len(samplelist24)

['062019_BEL_CBC_T1_16_MCAV', '062019_BEL_CBC_T1_17_MCAV', '062019_BEL_CBC_T1_20_MCAV', '062019_BEL_CBC_T1_22_MCAV', '062019_BEL_CBC_T1_24_MCAV', '062019_BEL_CBC_T1_3_MCAV', '062019_BEL_CBC_T1_4_MCAV', '062019_BEL_CBC_T1_6_MCAV', '062019_BEL_CBC_T1_9_MCAV', '062019_BEL_CBC_T2_16_MCAV', '062019_BEL_CBC_T2_18_MCAV', '062019_BEL_CBC_T2_23_MCAV', '062019_BEL_CBC_T2_28_MCAV', '062019_BEL_CBC_T2_5_MCAV', '062019_BEL_CBC_T2_8_MCAV', '062019_BEL_CBC_T2_9_MCAV', '062019_BEL_CBC_T3_1_MCAV', '062019_BEL_CBC_T3_11_MCAV', '062019_BEL_CBC_T3_14_MCAV', '062019_BEL_CBC_T3_15_MCAV', '062019_BEL_CBC_T3_16_MCAV', '062019_BEL_CBC_T3_6_MCAV', '062019_BEL_CBC_T3_8_MCAV', '062019_BEL_CBC_T3_9_MCAV', '062019_BEL_CBC_T1_10_MMEA', '062019_BEL_CBC_T1_14_MMEA', '062019_BEL_CBC_T2_12_MMEA', '062019_BEL_CBC_T2_13_MMEA', '062019_BEL_CBC_T2_14_MMEA', '062019_BEL_CBC_T2_15_MMEA', '062019_BEL_CBC_T2_6_MMEA', '062019_BEL_CBC_T2_7_MMEA', '062019_BEL_CBC_T3_20_MMEA', '062019_BEL_CBC_T3_22_MMEA', '062019_BEL_CBC_T3_3_MMEA'

220

In [19]:
# why do they have diff number of samples:
missing_samples = [sample for sample in genohublist24 if sample not in samplelist24]
print("Sample IDs that are NOT in metagenomics tracker:", missing_samples)

# just missing negatives that were sequenced 

Sample IDs that are NOT in metagenomics tracker: ['7_3_Neg', '7_11_Neg']


In [20]:
cd 03172025

/scratch/workspace/brooke_sienkiewicz_student_uml_edu-brooke-belseq/03172025


In [21]:
# num of seq files 
!ls -1 *R1_001.fastq.gz | wc -l

226


In [22]:
# 6 extra samples?

In [23]:
# save num & name of seq files 
!ls -1 *R1_001.fastq.gz > sample_list

In [24]:
mv sample_list ../sample_list

In [25]:
cd ..

/scratch/workspace/brooke_sienkiewicz_student_uml_edu-brooke-belseq


In [26]:
# upload sample_list 
with open("sample_list", "r") as f:
    sample_list = [line.strip() for line in f] 

In [27]:
# remove file endings and added sample number from sequencer 
# regex replacement
sample_list = [re.sub(r'_S.*_R.*_001.fastq.gz','', sample) for sample in sample_list]
print(sample_list)
# seqs in unity 

['052022_BEL_CBC_T1_10_PSTR', '052022_BEL_CBC_T1_11_PSTR', '052022_BEL_CBC_T1_12_MCAV', '052022_BEL_CBC_T1_13_MCAV', '052022_BEL_CBC_T1_1_PAST', '052022_BEL_CBC_T1_34_PAST', '052022_BEL_CBC_T1_35_OANN', '052022_BEL_CBC_T1_39_MCAV', '052022_BEL_CBC_T1_40_MCAV', '052022_BEL_CBC_T1_41_OANN', '052022_BEL_CBC_T1_4_PSTR', '052022_BEL_CBC_T1_52_PAST', '052022_BEL_CBC_T1_53_PAST', '052022_BEL_CBC_T1_54_MCAV', '052022_BEL_CBC_T1_55_PSTR', '052022_BEL_CBC_T1_57_MCAV', '052022_BEL_CBC_T1_60_MCAV', '052022_BEL_CBC_T1_61_PAST', '052022_BEL_CBC_T1_62_MCAV', '052022_BEL_CBC_T1_63_OFAV', '052022_BEL_CBC_T1_70_MCAV', '052022_BEL_CBC_T2_10_MCAV', '052022_BEL_CBC_T2_11_PAST', '052022_BEL_CBC_T2_12_MCAV', '052022_BEL_CBC_T2_13_PSTR', '052022_BEL_CBC_T2_14_PSTR', '052022_BEL_CBC_T2_45_PSTR', '052022_BEL_CBC_T2_46_PSTR', '052022_BEL_CBC_T2_4_MCAV', '052022_BEL_CBC_T2_56_MCAV', '052022_BEL_CBC_T2_59_OFAV', '052022_BEL_CBC_T2_5_PAST', '052022_BEL_CBC_T2_60_PAST', '052022_BEL_CBC_T2_62_PAST', '052022_BEL_CBC_T

In [28]:
# why do we have 4 extra samples in seq list 
missing_samples = [sample for sample in sample_list if sample not in genohublist24]
print("Sample IDs that are NOT in genohublist that are in seq files:", missing_samples)

Sample IDs that are NOT in genohublist that are in seq files: ['102019_BEL_CBC_T2_30_PSTR_host', '102019_BEL_CBC_T2_31_PSTR_host', 'Negative_extract_11-2-24', 'Undetermined']


In [29]:
missing_samples = [sample for sample in sample_list if sample not in samplelist24]
print("Sample IDs that are NOT in seq file that are in metagenomics tracker:", missing_samples)

Sample IDs that are NOT in seq file that are in metagenomics tracker: ['102019_BEL_CBC_T2_30_PSTR_host', '102019_BEL_CBC_T2_31_PSTR_host', '7_11_Neg', '7_3_Neg', 'Negative_extract_11-2-24', 'Undetermined']


In [30]:
missing_samples = [sample for sample in samplelist24 if sample not in sample_list]
print("Sample IDs that are NOT in metagenomics tracker that are in seq files:", missing_samples)

Sample IDs that are NOT in metagenomics tracker that are in seq files: []


In [31]:
# what is missing in our files 
missing_samples = [sample for sample in genohublist24 if sample not in sample_list]
print("Sequences missing:", missing_samples)

Sequences missing: []


In [32]:
# # remove leading 0 and see if they are still missing 
# missing_samples_nolead0 = [re.sub(r'^0(\d+)', r'\1', sample) for sample in missing_samples]
# print(missing_samples_nolead0)

In [33]:
# # what is missing in our files 
# missing_samples = [sample for sample in missing_samples_nolead0 if sample not in sample_list]
# print("Sequences missing:", missing_samples)

# # We have them all! (just some formatting/naming issues)

# ## jul 18, 2025 - have renamed all with leading 0 

### Make sample table

In [34]:
sample_list[0:5]

['052022_BEL_CBC_T1_10_PSTR',
 '052022_BEL_CBC_T1_11_PSTR',
 '052022_BEL_CBC_T1_12_MCAV',
 '052022_BEL_CBC_T1_13_MCAV',
 '052022_BEL_CBC_T1_1_PAST']

In [35]:
len(sample_list)

226

In [36]:
# match to sample metadata 

# load 
metadata=pd.read_csv('//project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/metadata/CBC_samples.csv')

#filter for UML samples (rna later or etoh) 
metadata=metadata[
    (metadata['Sample_type'] == 'Core_EtOH') |
    (metadata['Sample_type'] == 'Core_RNAlater')
]

#match list to tubelabel_species 
matched_metadata = metadata[metadata['Tubelabel_species'].isin(sample_list)]

In [37]:
matched_metadata.head()

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,Time_processed,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes
33,122022,BEL,CBC,12/4/22,CBC30N,1,NaN,22,OANN,NaN,NaN,Core_EtOH,120,Diseased_Margin,NaN,122022_BEL_CBC_T1_120_OANN,Depleted_UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B3,NaN,NaN
39,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,22,OANN,NaN,NaN,Core_EtOH,136,Diseased_Tissue,NaN,122022_BEL_CBC_T1_136_OANN,Depleted_UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B3,NaN,NaN
139,52022,BEL,CBC,5/21/22,CBC30N,1,22,22,OANN,NaN,NaN,Core_EtOH,41,Healthy,newly added May 2022,052022_BEL_CBC_T1_41_OANN,Depleted_UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B3,NaN,NaN
256,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,12,PSTR,NaN,NaN,Core_EtOH,122,Healthy,NaN,122022_BEL_CBC_T1_122_PSTR,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN
261,122022,BEL,CBC,12/4/22,CBC30N,1,NaN,6,PSTR,NaN,NaN,Core_EtOH,132,Diseased_Tissue,NaN,122022_BEL_CBC_T1_132_PSTR,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN


In [101]:
# add colony ID - t# newtagnum species
matched_metadata = matched_metadata.copy()
matched_metadata['TransectNum_str'] = 'T' + matched_metadata['TransectNum'].astype(str)
matched_metadata['colony_id'] = matched_metadata[['TransectNum_str', 'NewTagNum', 'Species']].astype(str).agg('_'.join, axis=1)
matched_metadata.drop(columns='TransectNum_str', inplace=True)

In [102]:
matched_metadata.columns

Index(['Month_year', 'Country', 'Location', 'CollectionDate', 'Transect',
       'TransectNum', 'OldTagNum', 'NewTagNum', 'Species', 'Time_sampled',
       'Time_processed', 'Sample_type', 'SampleNum', 'Health_status',
       'Sampling_notes', 'Tubelabel_species', 'Sample_physical_location',
       'Extraction_physical_location', 'Date_sequenced', 'Notes', 'colony_id'],
      dtype='object')

In [103]:
matched_metadata[['Month_year','CollectionDate','Transect','TransectNum','NewTagNum',
                  'Species','SampleNum','Health_status','Sample_type','Tubelabel_species']]

,Month_year,CollectionDate,Transect,TransectNum,NewTagNum,Species,SampleNum,Health_status,Sample_type,Tubelabel_species
33,122022,12/4/22,CBC30N,1,22,OANN,120,Diseased_Margin,Core_EtOH,122022_BEL_CBC_T1_120_OANN
39,122022,12/2/22,CBC30N,1,22,OANN,136,Diseased_Tissue,Core_EtOH,122022_BEL_CBC_T1_136_OANN
139,52022,5/21/22,CBC30N,1,22,OANN,41,Healthy,Core_EtOH,052022_BEL_CBC_T1_41_OANN
256,122022,12/2/22,CBC30N,1,12,PSTR,122,Healthy,Core_EtOH,122022_BEL_CBC_T1_122_PSTR
261,122022,12/4/22,CBC30N,1,6,PSTR,132,Diseased_Tissue,Core_EtOH,122022_BEL_CBC_T1_132_PSTR
...,...,...,...,...,...,...,...,...,...,...
1129,62019,6/21/19,SR30N,2,68,PAST,2,Healthy,Core_EtOH,062019_BEL_CBC_T2_2_PAST
1141,62019,6/21/19,SR30N,2,59,MCAV,5,Healthy,Core_EtOH,062019_BEL_CBC_T2_5_MCAV
1144,62019,6/21/19,SR30N,2,330,MMEA,6,Healthy,Core_EtOH,062019_BEL_CBC_T2_6_MMEA
1146,62019,6/21/19,SR30N,2,344,MMEA,7,Healthy,Core_EtOH,062019_BEL_CBC_T2_7_MMEA


In [104]:
# make sample table 
# unique species, transect, health statuses, timepoint 
    # leaving monthyear as is for now 


In [105]:
matched_metadata['Month_year'].unique()

[122022, 52022, 102019, 62019]
Categories (4, int64): [62019 < 102019 < 52022 < 122022]

In [106]:
# combine 062019 and 102019? 

In [107]:
month_order = [62019, 102019, 52022, 122022]
matched_metadata['Month_year'] = pd.Categorical(
    matched_metadata['Month_year'], categories=month_order, ordered=True
)

In [108]:
summary_table = (
    matched_metadata
    .groupby(['Month_year', 'Transect', 'Species', 'Health_status'])
    .size()
    .reset_index(name='n')
    .pivot_table(index=['Month_year', 'Transect', 'Health_status'],
                 columns='Species',
                 values='n',
                 fill_value=0)
    .astype(int)
)

summary_table.index = summary_table.index.set_names(['Month_year', 'Transect', 'Health_status'])
summary_table = summary_table.reset_index()

summary_table = summary_table.set_index('Month_year')
summary_table.columns.name = None  # removes 'Species' label above columns

# remove rows with all 0s
species_cols = ['MCAV', 'MMEA', 'OANN', 'OFAV', 'PAST', 'PSTR']
summary_table = summary_table.loc[~(summary_table[species_cols] == 0).all(axis=1)]

/tmp/ipykernel_3559972/3477100914.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['Month_year', 'Transect', 'Species', 'Health_status'])
/tmp/ipykernel_3559972/3477100914.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(index=['Month_year', 'Transect', 'Health_status'],


In [109]:
summary_table

,Transect,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
Month_year,,,,,,,,
62019,CBC30N,Healthy,9,2,0,0,5,0
62019,Lagoon,Healthy,8,5,0,0,9,0
62019,SR30N,Healthy,7,6,0,0,6,0
102019,CBC30N,Healthy,0,0,0,0,0,7
102019,Lagoon,Healthy,0,0,0,0,0,7
102019,SR30N,Healthy,0,0,0,0,0,7
52022,CBC30N,Diseased_Margin,3,0,0,0,0,1
52022,CBC30N,Diseased_Tissue,3,0,0,0,1,1
52022,CBC30N,Healthy,3,0,2,1,4,2


In [110]:
# just pre- and post- disease for each sp 

# group 2019s and 2022s
table = summary_table.copy()
table['Year'] = table.index.astype(str).str[-4:].astype(int)

condensed = (
    table
    .groupby(['Year', 'Health_status'])
    .sum(numeric_only=True)
    .reset_index()
)
condensed

,Year,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
0,2019,Healthy,24,13,0,0,20,21
1,2022,Diseased_Margin,9,0,2,2,4,6
2,2022,Diseased_Tissue,11,0,3,2,5,6
3,2022,Healthy,24,0,8,15,24,21


#### investigate colony numbers
- why are there 24 healthy mcav samples in 2019 AND 2022? 

In [111]:
# crosscheck healthy in 2019 and 2022 

# start with mcav subset 
mcav=matched_metadata[matched_metadata['Species']=="MCAV"]
# list of samples in 2019 with healthy
meta_2019=mcav[
    (mcav['Month_year']==62019) |
    (mcav['Month_year']==102019)]
ids_2019=set(meta_2019['colony_id'].unique())
# list of samples in 2022 with healthy 
meta_2022=mcav[
    (mcav['Month_year']==52022) |
    (mcav['Month_year']==122022)]
ids_2022=set(meta_2022['colony_id'].unique())
# do they match 
ids_2019 == ids_2022  # True if identical colonies

False

In [115]:
# colonies present in both years
print(len(ids_2019 & ids_2022))
print(len(ids_2019))
print(len(ids_2022))

17
24
24


In [123]:
# colonies not in 2022 
print('colonies not in 2022:',len(ids_2019 - ids_2022), 'colonies',
      ids_2019 - ids_2022)
# manually checking fate 
# t1: 324, 333, 329, 355, all died in 052022
# t2: 56 died in 052022
# t3: 9 & 12 died in 052022

# new colonies in 2022 not present in 2019
print('new colonies in 2022 not present in 2019:',len(ids_2022 - ids_2019), 'colonies',
      ids_2022 - ids_2019)
# manually checking fate 
# t3: 67 & 71 tagged in 122022
# all t4 tagged in 122022

colonies not in 2022: 7 colonies {'T1_342_MCAV', 'T1_333_MCAV', 'T1_329_MCAV', 'T1_355_MCAV', 'T2_56_MCAV', 'T3_9_MCAV', 'T3_12_MCAV'}
new colonies in 2022 not present in 2019: 7 colonies {'T4_28_MCAV', 'T4_30_MCAV', 'T3_71_MCAV', 'T4_95_MCAV', 'T4_94_MCAV', 'T3_67_MCAV', 'T4_76_MCAV'}


In [100]:
condensed2 = (
    table
    .groupby(['Year', 'Transect','Health_status']) 
    .sum(numeric_only=True)
    .reset_index()
)
condensed2

,Year,Transect,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
0,2019,CBC30N,Healthy,9,2,0,0,5,7
1,2019,Lagoon,Healthy,8,5,0,0,9,7
2,2019,SR30N,Healthy,7,6,0,0,6,7
3,2022,CBC30N,Diseased_Margin,3,0,1,0,1,2
4,2022,CBC30N,Diseased_Tissue,4,0,1,0,2,2
5,2022,CBC30N,Healthy,5,0,4,1,8,3
6,2022,CURLEW,Diseased_Margin,2,0,0,1,0,2
7,2022,CURLEW,Diseased_Tissue,2,0,0,1,0,2
8,2022,CURLEW,Healthy,3,0,0,3,0,3
9,2022,Lagoon,Diseased_Margin,4,0,0,0,2,0
